### **Cell 1: Environment Setup & API Key**

In [ ]:
# 1. INSTALL DEPENDENCIES
# Using the latest packages to ensure compatibility
!pip install langchain langchain-openai langchain-community langchain-classic langchain-text-splitters wikipedia numexpr faiss-cpu

import os
from getpass import getpass

# 2. API SETUP
# We use getpass so the API key is never hardcoded or exposed on GitHub
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

### **Cell 2: Basic LLM Call & Prompt Templates**

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate

# Initialize the Chat Model
chat_model = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)

print("--- 1. BASIC LLM CALL ---")
response = chat_model.invoke([HumanMessage(content="Explain quantum computing in one sentence.")])
print(response.content, "\n")

print("--- 2. PROMPT TEMPLATE ---")
# Create a reusable template
template = "You are an expert data scientist. Explain the concept of {concept} simply to a {target_audience}."
prompt = PromptTemplate.from_template(template)

formatted_prompt = prompt.format(concept="Random Forest", target_audience="10-year-old")
print(formatted_prompt)

### **Cell 3: Simple Chain (Using LCEL)**

In [ ]:
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain

memory = ConversationBufferMemory()
conversation = ConversationChain(
    llm=chat_model,
    memory=memory,
    verbose=False
)

print("--- Memory Example ---")
print("Turn 1:", conversation.predict(input="Hi, my name is Alex and I love Python."))
print("Turn 2:", conversation.predict(input="What is my name and my favorite programming language?"))

### **Cell 4: Memory Implementation**

In [ ]:
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain

print("--- 4. MEMORY IMPLEMENTATION ---")
# Initialize memory to store conversation history
memory = ConversationBufferMemory()

# Create a conversation chain that utilizes this memory
conversation = ConversationChain(
    llm=chat_model,
    memory=memory,
    verbose=False
)

# Turn 1: Providing information
print("Turn 1:", conversation.predict(input="Hi, my name is Alex and I love Python."))

# Turn 2: Testing if the LLM remembers the previous turn
print("Turn 2:", conversation.predict(input="What is my name and my favorite programming language?"))

### **Cell 5: Agents and Tools**

In [ ]:
from langchain_community.agent_toolkits.load_tools import load_tools
from langchain.agents import initialize_agent, AgentType

print("--- 5. AGENT WITH TOOLS ---")
# Equip the agent with external tools: Wikipedia for search, llm-math for calculations
tools = load_tools(["wikipedia", "llm-math"], llm=chat_model)

# Initialize a Zero-Shot ReAct agent
agent = initialize_agent(
    tools,
    chat_model,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# The agent will decide to search Wikipedia for the birth year, then use math to multiply it
agent.run("When was Albert Einstein born? Multiply that year by 5.")

### **Cell 6: Document Loaders & Vector Store (RAG)**

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter

print("--- 6. DOCUMENT LOADERS & VECTOR STORE ---")

# Step A: Auto-generate a dummy file so the code is guaranteed to be runnable without missing file errors
with open("company_policy.txt", "w") as f:
    f.write("The company allows remote work on Tuesdays and Thursdays. Office hours are 9 AM to 5 PM.")

# Step B: Load the Document
loader = TextLoader("company_policy.txt")
documents = loader.load()

# Step C: Split text into manageable chunks
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
docs = text_splitter.split_documents(documents)

# Step D: Create Embeddings and store them in a FAISS Vector Database
embeddings = OpenAIEmbeddings()
db = FAISS.from_documents(docs, embeddings)

# Step E: Perform a semantic search query against the vector store
query = "What is the work from home policy?"
search_results = db.similarity_search(query)

print("Search Query:", query)
print("Result Found:", search_results[0].page_content)